# Phase 3 - Process Mining

This notebook performs process-mining analysis on the synthetic P2P event log and case summary. It does not modify files under `data/raw/` and does not train machine-learning models.

In [ ]:
from pathlib import Path
from collections import Counter
import html
import importlib.util

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "raw").exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
EVENT_LOG_PATH = RAW_DIR / "p2p_event_log.csv"
CASE_SUMMARY_PATH = RAW_DIR / "p2p_case_summary.csv"

## 1. Load Raw Event Log and Case Summary

The raw CSVs are loaded read-only. Derived figures are written under `reports/figures/`.

In [ ]:
events = pd.read_csv(EVENT_LOG_PATH)
cases = pd.read_csv(CASE_SUMMARY_PATH)
events["timestamp_dt"] = pd.to_datetime(events["timestamp"], errors="coerce")
cases["start_dt"] = pd.to_datetime(cases["start_time"], errors="coerce")
cases["end_dt"] = pd.to_datetime(cases["end_time"], errors="coerce")

pd.DataFrame([
    {"dataset": "p2p_event_log.csv", "rows": len(events), "columns": events.drop(columns=["timestamp_dt"]).shape[1], "path_exists": EVENT_LOG_PATH.exists()},
    {"dataset": "p2p_case_summary.csv", "rows": len(cases), "columns": cases.drop(columns=["start_dt", "end_dt"]).shape[1], "path_exists": CASE_SUMMARY_PATH.exists()},
])

## 2. PM4Py Availability

PM4Py is optional here. If unavailable, the notebook uses a pandas directly-follows process map.

In [ ]:
pd.DataFrame({"package": ["pm4py"], "available": [importlib.util.find_spec("pm4py") is not None]})

## 3. Prepare Ordered Event Log

In [ ]:
events_sorted = events.sort_values(["case_id", "timestamp_dt", "event_id"]).copy()
events_sorted["next_activity"] = events_sorted.groupby("case_id")["activity"].shift(-1)
events_sorted["next_timestamp"] = events_sorted.groupby("case_id")["timestamp_dt"].shift(-1)
events_sorted["elapsed_hours_to_next"] = (events_sorted["next_timestamp"] - events_sorted["timestamp_dt"]).dt.total_seconds() / 3600
events_sorted["prev_activity"] = events_sorted.groupby("case_id")["activity"].shift(1)
events_sorted["prev_timestamp"] = events_sorted.groupby("case_id")["timestamp_dt"].shift(1)
events_sorted["hours_since_prev"] = (events_sorted["timestamp_dt"] - events_sorted["prev_timestamp"]).dt.total_seconds() / 3600
events_sorted.head()

## 4. Process Map and Transition Frequency

In [ ]:
process_edges = (
    events_sorted.dropna(subset=["next_activity"])
    .groupby(["activity", "next_activity"])
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)
process_edges

In [ ]:
def esc(value):
    return html.escape(str(value), quote=True)

def write_bar_svg(path, labels, values, title, x_label="", color="#1f77b4", value_fmt="{:.1f}", width=920, height=460):
    labels = [str(x) for x in labels]
    values = [0.0 if pd.isna(x) else float(x) for x in values]
    ml, mr, mt, mb = 260, 42, 58, 44
    ph, pw = height - mt - mb, width - ml - mr
    row_h = ph / max(len(labels), 1)
    max_v = max(max(values or [1.0]), 1.0)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="#fff"/>']
    parts.append(f'<text x="{width/2}" y="28" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{esc(title)}</text>')
    for i, (label, value) in enumerate(zip(labels, values)):
        y = mt + i * row_h + row_h * 0.18
        bh = max(row_h * 0.55, 8)
        bw = value / max_v * pw
        shown = label if len(label) <= 42 else label[:39] + "..."
        parts.append(f'<text x="{ml-10}" y="{y+bh*.72}" text-anchor="end" font-family="Arial" font-size="11">{esc(shown)}</text>')
        parts.append(f'<rect x="{ml}" y="{y}" width="{bw:.2f}" height="{bh:.2f}" fill="{color}" opacity="0.86"/>')
        parts.append(f'<text x="{ml+bw+6}" y="{y+bh*.72}" font-family="Arial" font-size="11">{esc(value_fmt.format(value))}</text>')
    parts.append(f'<text x="{ml+pw/2}" y="{height-10}" text-anchor="middle" font-family="Arial" font-size="12">{esc(x_label)}</text></svg>')
    Path(path).write_text("\n".join(parts), encoding="utf-8")

def write_grouped_bar_svg(path, labels, series, title):
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    width, height = 920, 460
    ml, mr, mt, mb = 230, 60, 70, 48
    ph, pw = height - mt - mb, width - ml - mr
    labels = [str(x) for x in labels]
    vals = {k: [float(x) for x in v] for k, v in series.items()}
    row_h = ph / max(len(labels), 1)
    max_v = max([max(v) for v in vals.values()] or [1.0])
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="#fff"/>']
    parts.append(f'<text x="{width/2}" y="28" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{esc(title)}</text>')
    for j, name in enumerate(vals):
        x = ml + j * 210
        parts.append(f'<rect x="{x}" y="45" width="13" height="13" fill="{palette[j]}"/><text x="{x+18}" y="56" font-family="Arial" font-size="12">{esc(name)}</text>')
    for i, label in enumerate(labels):
        y0 = mt + i * row_h
        parts.append(f'<text x="{ml-10}" y="{y0+row_h*.62}" text-anchor="end" font-family="Arial" font-size="11">{esc(label)}</text>')
        for j, name in enumerate(vals):
            y = y0 + row_h * (0.18 + j * 0.32)
            bh = row_h * 0.28
            bw = vals[name][i] / max(max_v, 1.0) * pw
            parts.append(f'<rect x="{ml}" y="{y:.2f}" width="{bw:.2f}" height="{bh:.2f}" fill="{palette[j]}" opacity="0.86"/><text x="{ml+bw+5}" y="{y+bh*.78:.2f}" font-family="Arial" font-size="10">{vals[name][i]:.1f}</text>')
    parts.append("</svg>")
    Path(path).write_text("\n".join(parts), encoding="utf-8")

def write_process_map_svg(path, edges):
    coords = {"Purchase Request": (80, 170), "Budget Review": (260, 70), "Manager Approval": (300, 170), "Purchase Order": (500, 170), "Vendor Confirmation": (710, 170), "Goods Receipt": (710, 330), "Invoice Verification": (500, 330), "Payment": (300, 330)}
    max_f = edges["frequency"].max()
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="960" height="430" viewBox="0 0 960 430">', '<rect width="100%" height="100%" fill="#fff"/>', '<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#555"/></marker></defs>', '<text x="480" y="28" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">Directly-Follows Process Map by Transition Frequency</text>']
    for _, row in edges.sort_values("frequency").iterrows():
        a, b, f = row["activity"], row["next_activity"], row["frequency"]
        x1, y1 = coords[a]; x2, y2 = coords[b]
        sw = 1.5 + 7 * f / max_f
        if a == b:
            parts.append(f'<path d="M{x1+45},{y1-22} C{x1+110},{y1-85} {x1+160},{y1-10} {x1+48},{y1+8}" fill="none" stroke="#d62728" stroke-width="{sw:.2f}" opacity="0.62" marker-end="url(#arrow)"/>')
            tx, ty = x1 + 100, y1 - 42
        else:
            parts.append(f'<line x1="{x1+58}" y1="{y1}" x2="{x2-58}" y2="{y2}" stroke="#555" stroke-width="{sw:.2f}" opacity="0.62" marker-end="url(#arrow)"/>')
            tx, ty = (x1 + x2) / 2, (y1 + y2) / 2 - 8
        parts.append(f'<text x="{tx}" y="{ty}" text-anchor="middle" font-family="Arial" font-size="10">{int(f):,}</text>')
    for name, (x, y) in coords.items():
        parts.append(f'<rect x="{x-66}" y="{y-22}" width="132" height="44" rx="5" fill="#f7f9fb" stroke="#2f5d7c"/><text x="{x}" y="{y+4}" text-anchor="middle" font-family="Arial" font-size="11">{esc(name)}</text>')
    parts.append("</svg>")
    Path(path).write_text("\n".join(parts), encoding="utf-8")

write_process_map_svg(FIG_DIR / "process_transition_frequency.svg", process_edges)
FIG_DIR / "process_transition_frequency.svg"

![Process transition frequency](../reports/figures/process_transition_frequency.svg)

## 5. Process Variants and SLA Breach Rates

In [ ]:
case_sequences = events_sorted.groupby("case_id")["activity"].agg(tuple)
variant_counts = case_sequences.value_counts()
variant_rank = {sequence: rank for rank, sequence in enumerate(variant_counts.index, start=1)}
case_variants = case_sequences.rename("sequence").reset_index()
case_variants["variant_rank"] = case_variants["sequence"].map(variant_rank)
case_variants["activity_sequence"] = case_variants["sequence"].apply(lambda sequence: " -> ".join(sequence))
case_variants = case_variants.merge(cases[["case_id", "sla_breached", "duration_days", "duration_hours", "priority", "category", "vendor_id"]], on="case_id")
variant_summary = (
    case_variants.groupby(["variant_rank", "activity_sequence"])
    .agg(cases=("case_id", "count"), sla_breaches=("sla_breached", "sum"), sla_breach_rate=("sla_breached", "mean"), avg_duration_days=("duration_days", "mean"), median_duration_days=("duration_days", "median"))
    .reset_index()
    .sort_values("cases", ascending=False)
)
variant_summary["case_pct"] = (variant_summary["cases"] / len(cases) * 100).round(2)
variant_summary["sla_breach_rate_pct"] = (variant_summary["sla_breach_rate"] * 100).round(2)
variant_summary[["variant_rank", "cases", "case_pct", "sla_breaches", "sla_breach_rate_pct", "avg_duration_days", "median_duration_days", "activity_sequence"]].head(10)

In [ ]:
write_bar_svg(FIG_DIR / "top_process_variants.svg", [f"Variant {int(row.variant_rank)}" for row in variant_summary.head(10).itertuples()], variant_summary.head(10)["case_pct"], "Top Process Variants by Case Share", "Percent of cases", value_fmt="{:.2f}%")
FIG_DIR / "top_process_variants.svg"

![Top process variants](../reports/figures/top_process_variants.svg)

## 6. Transition Timing

In [ ]:
transition_summary = (
    events_sorted.dropna(subset=["next_activity"])
    .groupby(["activity", "next_activity"])
    .agg(occurrences=("case_id", "count"), avg_elapsed_hours=("elapsed_hours_to_next", "mean"), median_elapsed_hours=("elapsed_hours_to_next", "median"), p95_elapsed_hours=("elapsed_hours_to_next", lambda series: series.quantile(0.95)))
    .reset_index()
)
transition_summary["from_to"] = transition_summary["activity"] + " -> " + transition_summary["next_activity"]
transition_summary = transition_summary.sort_values("avg_elapsed_hours", ascending=False)
transition_summary[["from_to", "occurrences", "avg_elapsed_hours", "median_elapsed_hours", "p95_elapsed_hours"]].round(2)

In [ ]:
write_bar_svg(FIG_DIR / "transition_duration_chart.svg", transition_summary.head(10)["from_to"], transition_summary.head(10)["avg_elapsed_hours"], "Slowest Transitions by Average Elapsed Hours", "Average elapsed hours", color="#d62728", value_fmt="{:.1f} h")
FIG_DIR / "transition_duration_chart.svg"

![Transition duration chart](../reports/figures/transition_duration_chart.svg)

## 7. Activity-Level Performance

In [ ]:
activity_performance = (
    events_sorted.groupby("activity")
    .agg(frequency=("case_id", "count"), avg_hours_since_prev=("hours_since_prev", "mean"), median_hours_since_prev=("hours_since_prev", "median"), p95_hours_since_prev=("hours_since_prev", lambda series: series.dropna().quantile(0.95) if series.notna().any() else np.nan))
    .reset_index()
    .sort_values("avg_hours_since_prev", ascending=False, na_position="last")
)
activity_performance.round(2)

## 8. Rework Analysis

In [ ]:
repeat_case_flags = case_sequences.apply(lambda sequence: len(sequence) != len(set(sequence))).rename("has_rework").reset_index()
rework_cases = cases.merge(repeat_case_flags, on="case_id")
rework_summary = (
    rework_cases.groupby("has_rework")
    .agg(cases=("case_id", "count"), avg_duration_days=("duration_days", "mean"), median_duration_days=("duration_days", "median"), sla_breaches=("sla_breached", "sum"), sla_breach_rate=("sla_breached", "mean"))
    .reset_index()
)
rework_summary["case_pct"] = (rework_summary["cases"] / len(cases) * 100).round(2)
rework_summary["sla_breach_rate_pct"] = (rework_summary["sla_breach_rate"] * 100).round(2)
rework_summary

In [ ]:
repeat_case_counter = Counter()
extra_repeat_counter = Counter()
for sequence in case_sequences:
    counts = Counter(sequence)
    for activity, count in counts.items():
        if count > 1:
            repeat_case_counter[activity] += 1
            extra_repeat_counter[activity] += count - 1
repeat_activity_summary = pd.DataFrame({"activity": list(repeat_case_counter.keys()), "cases_with_repeat": list(repeat_case_counter.values()), "extra_repetitions": [extra_repeat_counter[activity] for activity in repeat_case_counter]}).sort_values("cases_with_repeat", ascending=False)
repeat_activity_summary

In [ ]:
ordered_rework = rework_summary.sort_values("has_rework")
write_grouped_bar_svg(FIG_DIR / "rework_analysis.svg", ["No rework", "Rework"], {"Avg duration days": ordered_rework["avg_duration_days"], "SLA breach rate %": ordered_rework["sla_breach_rate"] * 100}, "Rework Impact on Duration and SLA Breach Rate")
FIG_DIR / "rework_analysis.svg"

![Rework analysis](../reports/figures/rework_analysis.svg)

## 9. Vendor Analysis

In [ ]:
purchase_to_vendor = events_sorted[(events_sorted["activity"] == "Purchase Order") & (events_sorted["next_activity"] == "Vendor Confirmation")].copy()
vendor_confirmation = (
    purchase_to_vendor.groupby("vendor_id")
    .agg(vendor_confirmation_cases=("case_id", "nunique"), avg_vendor_confirmation_hours=("elapsed_hours_to_next", "mean"), median_vendor_confirmation_hours=("elapsed_hours_to_next", "median"), p95_vendor_confirmation_hours=("elapsed_hours_to_next", lambda series: series.quantile(0.95)))
    .reset_index()
)
vendor_sla = (
    cases.groupby("vendor_id")
    .agg(total_cases=("case_id", "count"), sla_breaches=("sla_breached", "sum"), sla_breach_rate=("sla_breached", "mean"), avg_duration_days=("duration_days", "mean"), median_duration_days=("duration_days", "median"))
    .reset_index()
)
adequate_vendor_threshold = 200
vendor_performance = vendor_sla.merge(vendor_confirmation, on="vendor_id", how="left")
vendor_performance["sample_size_flag"] = np.where(vendor_performance["total_cases"] >= adequate_vendor_threshold, "adequate_sample", "small_sample")
vendor_performance["vendor_confirmation_zscore"] = (vendor_performance["avg_vendor_confirmation_hours"] - vendor_performance["avg_vendor_confirmation_hours"].mean()) / vendor_performance["avg_vendor_confirmation_hours"].std(ddof=0)
vendor_response_top = vendor_performance[vendor_performance["total_cases"] >= adequate_vendor_threshold].sort_values("avg_vendor_confirmation_hours", ascending=False)
vendor_response_top.head(10).round(2)

In [ ]:
vendor_sla_top = vendor_performance[vendor_performance["total_cases"] >= adequate_vendor_threshold].sort_values("sla_breach_rate", ascending=False)
vendor_sla_top.head(10).round(2)

In [ ]:
pd.DataFrame({"metric": ["vendor_response_vs_sla_breach_rate_correlation"], "value": [vendor_performance["avg_vendor_confirmation_hours"].corr(vendor_performance["sla_breach_rate"])]})

In [ ]:
write_grouped_bar_svg(FIG_DIR / "vendor_performance.svg", vendor_response_top.head(10)["vendor_id"], {"Avg PO to Vendor Confirmation hours": vendor_response_top.head(10)["avg_vendor_confirmation_hours"], "SLA breach rate %": vendor_response_top.head(10)["sla_breach_rate"] * 100}, "Vendor Confirmation Timing and SLA Rate - Adequate Samples")
FIG_DIR / "vendor_performance.svg"

![Vendor performance](../reports/figures/vendor_performance.svg)

## 10. Priority and Category Analysis

In [ ]:
priority_summary = (
    cases.groupby("priority")
    .agg(cases=("case_id", "count"), avg_duration_days=("duration_days", "mean"), median_duration_days=("duration_days", "median"), p95_duration_days=("duration_days", lambda series: series.quantile(0.95)), sla_breaches=("sla_breached", "sum"), sla_breach_rate=("sla_breached", "mean"))
    .reset_index()
    .sort_values("avg_duration_days", ascending=False)
)
priority_summary["sla_breach_rate_pct"] = (priority_summary["sla_breach_rate"] * 100).round(2)
priority_summary.round(2)

In [ ]:
category_summary = (
    cases.groupby("category")
    .agg(cases=("case_id", "count"), avg_duration_days=("duration_days", "mean"), median_duration_days=("duration_days", "median"), p95_duration_days=("duration_days", lambda series: series.quantile(0.95)), sla_breaches=("sla_breached", "sum"), sla_breach_rate=("sla_breached", "mean"))
    .reset_index()
    .sort_values("avg_duration_days", ascending=False)
)
category_summary["sla_breach_rate_pct"] = (category_summary["sla_breach_rate"] * 100).round(2)
category_summary.round(2)

In [ ]:
write_bar_svg(FIG_DIR / "sla_breach_comparison.svg", category_summary["category"], category_summary["sla_breach_rate"] * 100, "SLA Breach Rate by Category", "SLA breach rate (%)", color="#ff7f0e", value_fmt="{:.1f}%")
FIG_DIR / "sla_breach_comparison.svg"

![SLA breach comparison](../reports/figures/sla_breach_comparison.svg)

## 11. Feature Engineering Candidates for Later SLA-Breach Prediction

No model is trained here. These are candidate feature families to evaluate later with leakage controls.

In [ ]:
pd.DataFrame({
    "feature_family": ["variant_family", "directly_follows_counts", "elapsed_transition_times", "budget_review_flag", "rework_flags", "activity_repeat_counts", "vendor_response_history", "vendor_sla_history_smoothed", "priority_category_purchase_amount", "early_cycle_elapsed_time", "bottleneck_exposure_flags"],
    "why_consider": ["Major variants show different SLA breach rates.", "Transition structure captures process path deviations.", "Slow transitions are associated with longer cycle time.", "Budget Review variants show elevated SLA breach rates.", "Rework cases have higher duration and breach rates.", "Repeated Invoice Verification, Manager Approval, and Vendor Confirmation are concentrated rework signals.", "Vendor Confirmation response varies substantially by vendor.", "Historical vendor behavior may be predictive if computed without leakage.", "Priority and category are associated with different durations and breach rates.", "Early elapsed time may indicate emerging delay risk.", "Exposure to slow PO-to-vendor or vendor-to-receipt gaps may flag risk."],
    "leakage_note": ["Use only sequence known at prediction time.", "Use only completed transitions available at prediction time.", "Do not use future transition durations.", "Safe only after Budget Review is observed or known.", "Safe only for repeats observed before prediction.", "Use counts observed before prediction.", "Compute from prior cases or training folds only.", "Smooth and compute out-of-fold to avoid target leakage.", "Usually safe if known at case start.", "Define a fixed prediction checkpoint.", "Avoid using full-case duration or post-outcome information."],
})